# Examen Final
## Probabilidad y Estadística

Legajo a2412

Quiroga, Martin Gabriel

Siguiendo con la historia de Don Francisco, con el tiempo y gracias a los análisis de Matías, el pequeño comerciante de barrio cuenta hoy con 5 supermercados: ’Santa Ana’, ’La Floresta’, ’Los Cedros’, ’Palermo’ y ’Córdoba’.
También Matías ha avanzado en la Especialización en Inteligencia Artificial. Un día Don Francisco le plantea algunas inquietudes adicionales:
1. Don Francisco quiere entender mejor las ventas por mes del supermercado ’Santa Ana’.
2. Más aún, Don Francisco no sabe si puede estar seguro de que las ventas son las mismas en todos los supermercados o si hay alguno que se comporte mejor que los demás, y si alguna de las tiendas necesita más atención porque sus ventas sean peores que las de las otras.

Con base en lo anterior,

1. (2.5 puntos) Crear una simulación de las ventas diarias de los almacenes de Don Francisco y de Don Miguel, usando distribuciones Poisson, entre los años 2023, 2024 y 2025. En cada fecha, el parámetro $\lambda_t$ debe ser la suma de los siguientes efectos:

**Efecto anual:**
| Año | Efecto |
| --- | --- |
| 2023 | 1000 |
| 2024 | 1500 |
| 2025 | 2000 |

**Efecto mensual:**
| Mes | Efecto |
| --- | --- |
| Enero | 1000 |
| Febrero | 1500 |
| Marzo | 2000 |
| Abril | 2000 |
| Mayo | 2500 |
| Junio | 2500 |
| Julio | 3000 |
| Agosto | 2500 |
| Septiembre | 2500 |
| Octubre | 2000 |
| Noviembre | 1500 |
| Diciembre | 1000 |

**Efecto diario:**
| Día | Efecto |
| --- | --- |
| Domingo | 1000 |
| Lunes | 2000 |
| Martes | 3000 |
| Miércoles | 3500 |
| Jueves | 3000 |
| Viernes | 2000 |
| Sábado | 1000 |

**Efecto por tienda:**
| Tienda | Efecto |
| --- | --- |
| Santa Ana | 5000 |
| La Floresta | 200 |
| Los Cedros | 3000 |
| Palermo | 1000 |
| Córdoba | 3000 |

2. (2.5 puntos) Con base en los datos generados, determinen intervalos de confianza empíricos para el supermercado ’Santa Ana’ en cada mes, para significancias del 95 % y el 99 %.
3. (2.5 puntos) De igual manera, realicen pruebas ANOVA para determinar si las ventas esperadas de todas las tiendas son iguales o no, con significancia del 95.
4. (2.5 puntos) Finalmente, identifiquen la tienda con mayor promedio de ventas y la tienda con menor promedio de ventas y realicen una prueba de hipótesis para determinar si la diferencia entre ellas es distinta de cero o no. Verifiquen si las tiendas identificadas corresponden a las tiendas con mayores y menores efectos.

In [29]:
import numpy as np
import pandas as pd
import datetime
from scipy.stats import poisson

#### 1) Crear una simulación de las ventas diarias de los almacenes de Don Francisco y de Don Miguel, usando distribuciones Poisson, entre los años 2023, 2024 y 2025.

In [30]:
# EL valor de lambda ahora dependerá de dos "variables": el día (en el que se
# incluye día de la semana, mes y año) así como de la tienda correspondiente, lo
# que nos llevará a tener que calcular un lambda por día, por almacén. Esto 
# sería n_días * n_tiendas = (365 + 366 + 365) * 5 = 5480 valores.
# Entre las posibles alternativas, se decide crear un único dataframe con los 
# 5480 registros.

start_date = datetime.datetime(2023, 1, 1)
end_date = datetime.datetime(2025, 12, 31)
fechas = pd.date_range(start_date, end_date)

tiendas = ['Santa Ana', 'La Floresta', 'Los Cedros', 'Palermo', 'Córdoba']

# Se crea un dataframe multi-índice (día y tienda)
df = pd.MultiIndex.from_product([fechas, tiendas], names=['date', 'store']).to_frame(index=False)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['dayOfWeek'] = df['date'].dt.day_of_week


In [31]:
# Se crean los diccionarios de efectos
year_effect_map = {
    2023: 1000,
    2024: 1500,
    2025: 2000
}
month_effect_map =  {
    1: 1000,
    2: 1500,
    3: 2000,
    4: 2000,
    5: 2500,
    6: 2500,
    7: 3000,
    8: 2500,
    9: 2500,
    10: 2000,
    11: 1500,
    12: 1000
}
# Usando pandas.DateTime el día 0 corresponde al Lunes y el 6 al Domingo
dow_effect_map = {
    6: 1000,
    0: 2000,
    1: 3000,
    2: 3500,
    3: 3000,
    4: 2000,
    5: 1000
}

store_effect_map = {
    'Santa Ana': 5000,
    'La Floresta': 2000,
    'Los Cedros': 3000,
    'Palermo': 1000,
    'Córdoba':3000
}


df['year_effect'] = df['year'].map(year_effect_map)
df['month_effect'] = df['month'].map(month_effect_map)
df['dow_effect'] = df['dayOfWeek'].map(dow_effect_map)
df['store_effect'] = df['store'].map(store_effect_map)

df['lambda'] = df['year_effect'] + df['month_effect'] + df['dow_effect'] + df['store_effect']

In [32]:
# Ahora sí, simulamos todas las ventas por día y tienda, siguiendo Poisson
rng = np.random.default_rng(seed=42)
df['sales'] = poisson.rvs(mu=df['lambda'], random_state=rng)

In [33]:
# Visualizamos los primeros registros del dataframe generado
df.head(20)

,date,store,year,month,dayOfWeek,year_effect,month_effect,dow_effect,store_effect,lambda,sales
0,2023-01-01,Santa Ana,2023,1,6,1000,1000,1000,5000,8000,8076
1,2023-01-01,La Floresta,2023,1,6,1000,1000,1000,2000,5000,5087
2,2023-01-01,Los Cedros,2023,1,6,1000,1000,1000,3000,6000,5878
3,2023-01-01,Palermo,2023,1,6,1000,1000,1000,1000,4000,4051
4,2023-01-01,Córdoba,2023,1,6,1000,1000,1000,3000,6000,5899
5,2023-01-02,Santa Ana,2023,1,0,1000,1000,2000,5000,9000,8965
6,2023-01-02,La Floresta,2023,1,0,1000,1000,2000,2000,6000,6032
7,2023-01-02,Los Cedros,2023,1,0,1000,1000,2000,3000,7000,6987
8,2023-01-02,Palermo,2023,1,0,1000,1000,2000,1000,5000,5011
9,2023-01-02,Córdoba,2023,1,0,1000,1000,2000,3000,7000,7089


#### 2) Con base en los datos generados, determinen intervalos de confianza empíricos para el supermercado ’Santa Ana’ en cada mes, para significancias del 95 % y el 99 %.

Se debe tomar solo la tienda de Santa Ana y agrupar los datos mensualmente. Luego, para cada mes se deben determinar los intervalos de confianza.

Se toman los datos como empíricos tal que no se conocen los estadísticos 
de la población (es decir, solo usamos los datos observados sin asumir la forma de la distribución subyacente), pero sí podemos obtener los de la muestra. Dentro de un mismo mes, las observaciones no provienen de una única distribución Poisson, sino de una mezcla: cada día tiene un lambda distinto porque varía el efecto del día de la semana y el efecto del año. Por lo tanto, no podemos asumir directamente un modelo paramétrico simple para la distribución mensual.

Por el teorema central del límite, como tendremos aproximadamente 90 datos para cada mes (~30 * 3 años), la distribución de la media muestral se arpoxima lo suficiente a una normal, sin importar la distribución original.


In [47]:
# Se filtra por tienda Santa Ana
df_sa = df[df['store'] == 'Santa Ana'].copy()

# Se agrupa por mes, y se calculan la media, la desviación estándar y la cantidad
# de observacions, para cada uno
sa_mes = df_sa.groupby('month')['sales'].agg(['mean', 'std', 'count']).reset_index()

# Además, se agrega el cálculo de s/sqrt(n) (error estándar) para simplificar
# otros cálculos después
sa_mes['se'] = sa_mes['std'] / np.sqrt(sa_mes['count'])
sa_mes


,month,mean,std,count,se
0,1,9748.752688,1042.957309,93,108.149635
1,2,10215.847059,997.283841,85,108.170620
2,3,10673.526882,976.078997,93,101.214677
3,4,10722.566667,1039.228950,90,109.544350
4,5,11252.612903,989.230630,93,102.578438
5,6,11174.344444,1001.938180,90,105.613557
6,7,11741.483871,1047.583810,93,108.629381
7,8,11220.397849,987.043578,93,102.351651
8,9,11179.733333,1027.860621,90,108.346023
9,10,10734.989247,1022.658019,93,106.044697


Al determinar media y desviación estándar usando los datos muestrales, para determinar los intervalos de confianza se dede usar la distribución de t-Student. Los intervalos a determinar son bilaterales, con $\alpha = 0.05$ y $\alpha = 0.01$, y un $n$ variable por mes, dependiendo de la cantidad de días del mismo.

Con Python, podemos obtener buscar el valor $t^*$ usando la librería de `scipy`.

In [49]:
from scipy.stats import t

sig1 = 0.95
sig2 = 0.99
alfa1 = 1 - sig1
alfa2 = 1 - sig2

# El número de observaciones n será para cada mes, se aplica directamente desde
# el dataframe. Calculamos el t* correspondiente para cada mes
sa_mes['t_95'] = t.ppf(1 - alfa1/2, df=sa_mes['count'] - 1)
sa_mes['t_99'] = t.ppf(1 - alfa2/2, df=sa_mes['count'] - 1)

# Luego, se calculan los límites inferior y superior de los intervalos de
# confianza para cada mes.
sa_mes['t_95_lower'] = sa_mes['mean'] - sa_mes['t_95'] * sa_mes['se']
sa_mes['t_95_upper'] = sa_mes['mean'] + sa_mes['t_95'] * sa_mes['se']
sa_mes['t_99_lower'] = sa_mes['mean'] - sa_mes['t_99'] * sa_mes['se']
sa_mes['t_99_lower'] = sa_mes['mean'] + sa_mes['t_99'] * sa_mes['se']

In [50]:
sa_mes

,month,mean,std,count,se,t_95,t_99,t_95_upper,t_95_lower,t_99_lower
0,1,9748.752688,1042.957309,93,108.149635,1.986086,2.630330,9963.547198,9533.958178,10033.221875
1,2,10215.847059,997.283841,85,108.170620,1.988610,2.635632,10430.956200,10000.737918,10500.945056
2,3,10673.526882,976.078997,93,101.214677,1.986086,2.630330,10874.547967,10472.505797,10939.754843
3,4,10722.566667,1039.228950,90,109.544350,1.986979,2.632204,10940.228956,10504.904377,11010.909763
4,5,11252.612903,989.230630,93,102.578438,1.986086,2.630330,11456.342535,11048.883271,11522.428005
5,6,11174.344444,1001.938180,90,105.613557,1.986979,2.632204,11384.196333,10964.492555,11452.340893
6,7,11741.483871,1047.583810,93,108.629381,1.986086,2.630330,11957.231198,11525.736544,12027.214947
7,8,11220.397849,987.043578,93,102.351651,1.986086,2.630330,11423.677063,11017.118636,11489.616428
8,9,11179.733333,1027.860621,90,108.346023,1.986979,2.632204,11395.014573,10964.452094,11464.922188
9,10,10734.989247,1022.658019,93,106.044697,1.986086,2.630330,10945.603168,10524.375326,11013.921753


Sea $X_{m,i}$ la venta diaria del supermercado Santa Ana en el día $i$-ésimo del mes $m$, donde:

$m \in {1, 2, \dots, 12}$ (mes del año)

$i \in {1, 2, \dots, n_m}$ (observaciones del mes $m$ a lo largo de los 3 años)

$$X_{m,i} \sim \text{Poisson}(\lambda_{m,i})$$

$$\bar{X}m = \frac{1}{n_m} \sum{i=1}^{n_m} X_{m,i}$$

$$S_m^2 = \frac{1}{n_m - 1} \sum_{i=1}^{n_m} (X_{m,i} - \bar{X}_m)^2$$

$$\frac{\bar{X}_m - \mu_m}{S_m / \sqrt{n_m}} \xrightarrow{d} \mathcal{N}(0, 1)$$
$$IC_m = \left[;\bar{X}m - z{\alpha/2} \cdot \frac{S_m}{\sqrt{n_m}} ;,; \bar{X}m + z{\alpha/2} \cdot \frac{S_m}{\sqrt{n_m}};\right]$$

$$P(\mu_m \in IC_m) \approx 1 - \alpha$$
